# Relatório experimental — MAC, MPC e controles clássicos

**Equação de Burgers | Aprendizado supervisionado | 5 sementes | 10.000 épocas**

Este notebook lê os resultados já calculados em `results/supervised/` e produz tabelas e figuras para o relatório científico. **Não executa novos treinamentos.**

**Roteiro**

1. Leitura e auditoria dos experimentos.
2. Desempenho: RMSE, erro relativo L2 e dispersão.
3. Comparações pareadas, semente a semente.
4. Convergência: curvas da função de perda e épocas até limiares.
5. Normas dos gradientes clássicos e auxiliares.
6. MPC com produto de correlatores e diferentes pesos de Pauli $k$.
7. Exportação das tabelas e figuras.

As figuras resumem **valores medidos** dos arquivos `history.csv` e `metrics.json`. Não utilizamos integrais ou uma métrica agregada artificial de convergência.

## 0. Preparação

Pode executar o notebook na raiz do projeto ou em `notebooks/`. Os resultados são salvos em `report/supervised/`. Bibliotecas: `pandas`, `numpy`, `matplotlib`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

BASE = ROOT / "results" / "supervised"
OUT = ROOT / "report" / "supervised"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

SEEDS = [1924, 1925, 1926, 1958, 1997]

NAMES = {
    "mac": "MAC",
    "mlp_mac": "MLP equivalente ao MAC",
    "mac_frozen": "MAC congelado",
    "mpc": "MPC (potências)",
    "mlp_mpc": "MLP equivalente à MPC",
    "mpc_frozen": "MPC congelada",
    "mpc_classical": "MPC clássica",
    "mpc_classical_frozen": "MPC clássica congelada",
    "mpc_linear": "MPC sem potências",
    "mlp_mpc_linear": "MLP equivalente à MPC linear",
    "mpc_linear_frozen": "MPC linear congelada",
    "mlp": "MLP convencional",
}

pd.set_option("display.max_rows", 100)
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110, "savefig.dpi": 180})

print("Resultados:", BASE)
print("Figuras e tabelas:", OUT)
assert BASE.exists(), "Pasta results/supervised não encontrada."

## 1. Leitura e auditoria

Cada execução deverá conter `metrics.json` e `history.csv`. A presença de `.complete` indica que o executor marcou a tarefa como concluída. A tabela de auditoria aponta sementes ou históricos ausentes sem inventar resultados.

In [ ]:
metrics = []
histories = []

for file in sorted(BASE.glob("**/metrics.json")):
    parts = file.relative_to(BASE).parts
    model = parts[0]
    k = next((int(part[2:]) for part in parts if part.startswith("k_")), None)
    name = f"{model}_k{k}" if k is not None else model
    seed = int(file.parent.name.replace("seed_", ""))
    data = json.loads(file.read_text())
    history_file = file.parent / "history.csv"

    metrics.append({
        "model": name,
        "label": f"MPC produto (k={k})" if k is not None else NAMES.get(model, model),
        "seed": seed,
        "k": k,
        "epochs": data.get("epochs"),
        "rmse": data.get("rmse"),
        "relative_l2": data.get("relative_l2"),
        "complete": (file.parent / ".complete").exists(),
        "history_exists": history_file.exists(),
    })

    if history_file.exists():
        h = pd.read_csv(history_file)
        h["model"] = name
        h["seed"] = seed
        h["k"] = k
        h["loss"] = h["train_mse"] if "train_mse" in h else h.get("total_loss", np.nan)
        histories.append(h)

m = pd.DataFrame(metrics)
assert not m.empty, "Nenhum metrics.json encontrado."
h = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()

m = m.sort_values(["model", "seed"]).reset_index(drop=True)
display(m[["label", "seed", "epochs", "complete", "history_exists", "rmse", "relative_l2"]])
print(f"Execuções encontradas: {len(m)} | Históricos encontrados: {len(histories)}")

In [ ]:
audit = (
    m.groupby(["model", "label"], as_index=False)
    .agg(seeds=("seed", "nunique"), finished=("complete", "sum"),
         histories=("history_exists", "sum"),
         epochs_min=("epochs", "min"), epochs_max=("epochs", "max"))
)
audit["missing_seeds"] = audit["model"].map(
    lambda model: str(sorted(set(SEEDS) - set(m.loc[m["model"] == model, "seed"])))
)
display(audit)
audit.to_csv(OUT / "audit.csv", index=False)

## 2. Desempenho final

Reportamos **média, mediana, desvio-padrão, mínimo e máximo** entre sementes. A mediana é importante porque uma execução com erro elevado pode distorcer bastante a média. Os resultados da MPC produto são apresentados separadamente na seção 6.

In [ ]:
regular = m[m["k"].isna()].copy()

summary = (
    regular.groupby(["model", "label"], as_index=False)
    .agg(n=("rmse", "count"),
         rmse_mean=("rmse", "mean"), rmse_median=("rmse", "median"),
         rmse_std=("rmse", "std"), rmse_min=("rmse", "min"),
         rmse_max=("rmse", "max"),
         l2_mean=("relative_l2", "mean"),
         l2_median=("relative_l2", "median"),
         l2_std=("relative_l2", "std"))
    .sort_values("rmse_median")
)
display(summary.round(7))
summary.to_csv(OUT / "performance_summary.csv", index=False)
m.to_csv(OUT / "performance_by_seed.csv", index=False)

In [ ]:
# Boxplot: RMSE por modelo (cinco pontos por caixa, quando disponíveis)
models = summary["model"].tolist()
values = [regular.loc[regular["model"] == model, "rmse"].dropna().values for model in models]

fig, ax = plt.subplots(figsize=(max(9, len(models) * 0.9), 5))
ax.boxplot(values, tick_labels=[NAMES.get(model, model) for model in models],
           showfliers=True)
for i, v in enumerate(values, 1):
    ax.scatter(np.full(len(v), i), v, s=24, zorder=3, alpha=0.7)
ax.set_yscale("log")
ax.set_ylabel("RMSE (escala log)")
ax.set_title("Distribuição do RMSE nas sementes")
ax.tick_params(axis="x", rotation=65)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG / "01_rmse_boxplot.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(max(9, len(models) * 0.9), 5))
values = [regular.loc[regular["model"] == model, "relative_l2"].dropna().values for model in models]
ax.boxplot(values, tick_labels=[NAMES.get(model, model) for model in models])
for i, v in enumerate(values, 1):
    ax.scatter(np.full(len(v), i), v, s=24, zorder=3, alpha=0.7)
ax.set_yscale("log")
ax.set_ylabel("Erro relativo L2 (escala log)")
ax.set_title("Distribuição do erro relativo L2")
ax.tick_params(axis="x", rotation=65)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG / "02_l2_boxplot.png", bbox_inches="tight")
plt.show()

## 3. Comparações pareadas entre arquiteturas

Comparamos os modelos **na mesma semente**, evitando comparar inicializações distintas. O ganho é positivo quando o primeiro modelo da dupla apresenta RMSE menor do que o segundo. As comparações de congelamento e coeficientes clássicos investigam parametrizações diferentes, não provam por si só uma vantagem quântica.

In [ ]:
PAIRS = [
    ("mac", "mlp_mac"),
    ("mpc", "mlp_mpc"),
    ("mpc", "mpc_frozen"),
    ("mpc", "mpc_classical"),
    ("mpc_classical", "mpc_classical_frozen"),
    ("mpc", "mpc_linear"),
    ("mpc_linear", "mlp_mpc_linear"),
    ("mpc_linear", "mpc_linear_frozen"),
]

comparisons = []
paired = []

for a, b in PAIRS:
    left = regular.loc[regular["model"] == a, ["seed", "rmse", "relative_l2"]]
    right = regular.loc[regular["model"] == b, ["seed", "rmse", "relative_l2"]]
    d = left.merge(right, on="seed", suffixes=("_a", "_b"))
    if d.empty:
        continue

    d["model_a"], d["model_b"] = a, b
    d["gain_percent"] = 100 * (d["rmse_b"] - d["rmse_a"]) / d["rmse_b"]
    paired.append(d)
    comparisons.append({
        "comparison": f"{a} vs {b}",
        "n_pairs": len(d),
        "wins_a": int((d["rmse_a"] < d["rmse_b"]).sum()),
        "median_gain_percent": d["gain_percent"].median(),
        "median_rmse_a": d["rmse_a"].median(),
        "median_rmse_b": d["rmse_b"].median(),
    })

comp = pd.DataFrame(comparisons)
display(comp.round(5))
comp.to_csv(OUT / "paired_comparisons.csv", index=False)
if paired:
    pd.concat(paired).to_csv(OUT / "paired_by_seed.csv", index=False)

In [ ]:
# Um gráfico pareado por comparação disponível
for a, b in PAIRS:
    x = regular.loc[regular["model"] == a, ["seed", "rmse"]]
    y = regular.loc[regular["model"] == b, ["seed", "rmse"]]
    d = x.merge(y, on="seed", suffixes=("_a", "_b"))
    if d.empty:
        continue

    fig, ax = plt.subplots(figsize=(5.2, 4.3))
    for _, row in d.iterrows():
        ax.plot([0, 1], [row["rmse_a"], row["rmse_b"]],
                "o-", alpha=0.75, label=f"Seed {int(row['seed'])}")
    ax.set_xticks([0, 1], [NAMES.get(a, a), NAMES.get(b, b)])
    ax.set_yscale("log")
    ax.set_ylabel("RMSE (escala log)")
    ax.set_title(f"{NAMES.get(a, a)} × {NAMES.get(b, b)}")
    ax.grid(axis="y", alpha=0.2)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / f"03_paired_{a}_vs_{b}.png", bbox_inches="tight")
    plt.show()

## 4. Convergência durante o treinamento

Usamos a **MSE de treinamento** registrada pelo `run.py` nas épocas 1, 100, 200, ..., 10.000. Cada linha abaixo representa a **mediana entre as sementes**; a faixa mostra o intervalo interquartil (25%–75%).

**Sem integral da curva.** Também calculamos a primeira época *registrada* que atingiu perdas inferiores a $10^{-3}$, $10^{-4}$ e $10^{-5}$. Como o histórico é amostrado a cada 100 épocas, os tempos até limiar são aproximados.

In [ ]:
GROUPS = {
    "mac": ["mac", "mlp_mac", "mac_frozen"],
    "mpc": ["mpc", "mlp_mpc", "mpc_frozen", "mpc_classical", "mpc_classical_frozen"],
    "linear": ["mpc_linear", "mlp_mpc_linear", "mpc_linear_frozen"],
    "product": [f"mpc_product_k{k}" for k in range(1, 6)],
}

assert not h.empty, "Nenhum history.csv encontrado."

for group, names in GROUPS.items():
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for name in names:
        d = h[h["model"] == name]
        if d.empty:
            continue
        g = d.groupby("epoch")["loss"]
        median = g.median().clip(lower=1e-15)
        q25 = g.quantile(0.25).clip(lower=1e-15)
        q75 = g.quantile(0.75).clip(lower=1e-15)
        line, = ax.plot(median.index, median.values,
                        label=NAMES.get(name, name.replace("mpc_product_k", "Produto k=")))
        ax.fill_between(median.index, q25.values, q75.values,
                        color=line.get_color(), alpha=0.12)
    if ax.lines:
        ax.set_yscale("log")
        ax.set_xlabel("Épocas")
        ax.set_ylabel("MSE de treinamento (escala log)")
        ax.set_title(f"Convergência — {group.upper()}")
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG / f"04_convergence_{group}.png", bbox_inches="tight")
        plt.show()
    else:
        plt.close(fig)

In [ ]:
# Arquivos individuais: uma figura com as cinco curvas de cada modelo
for model, d in h.groupby("model"):
    fig, ax = plt.subplots()
    for seed, ds in d.groupby("seed"):
        ax.plot(ds["epoch"], ds["loss"].clip(lower=1e-15),
                label=f"Seed {seed}", alpha=0.8)
    ax.set_yscale("log")
    ax.set_xlabel("Épocas")
    ax.set_ylabel("MSE de treinamento")
    ax.set_title(NAMES.get(model, model))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIG / f"04_individual_{model}.png", bbox_inches="tight")
    plt.close(fig)

print("Curvas individuais salvas em:", FIG)

In [ ]:
thresholds = [1e-3, 1e-4, 1e-5]
rows = []
for (model, seed), d in h.groupby(["model", "seed"]):
    for threshold in thresholds:
        reached = d.loc[d["loss"] <= threshold, "epoch"]
        rows.append({
            "model": model,
            "seed": seed,
            "threshold": threshold,
            "first_logged_epoch": reached.min() if not reached.empty else np.nan,
        })

thresholds_by_seed = pd.DataFrame(rows)
thresholds_summary = (
    thresholds_by_seed.groupby(["model", "threshold"], as_index=False)
    .agg(seeds_reached=("first_logged_epoch", "count"),
         median_epoch=("first_logged_epoch", "median"))
)
display(thresholds_summary)
thresholds_by_seed.to_csv(OUT / "convergence_thresholds_by_seed.csv", index=False)
thresholds_summary.to_csv(OUT / "convergence_thresholds_summary.csv", index=False)

## 5. Normas dos gradientes

O executor registrou a norma $L_2$ do gradiente **clássico** (`grad_classical`) e **auxiliar** (`grad_aux`), antes da atualização do Adam. Os gráficos mantêm esses dois grupos separados.

- **Clássico:** parâmetros da MLP.
- **Auxiliar:** parâmetros do circuito ou coeficientes clássicos treináveis.
- **Congelado / MLP pura:** gradiente auxiliar nulo por definição; isso **não** é evidência de gradiente evanescente.

Valores em zero não aparecem no eixo logarítmico das curvas auxiliares. A comparação direta de magnitudes deve ser feita com cautela, porque os grupos têm parametrizações e tamanhos diferentes.

In [ ]:
for column, title in [
    ("grad_classical", "Gradiente clássico"),
    ("grad_aux", "Gradiente auxiliar"),
]:
    if column not in h:
        print(f"Coluna ausente: {column}")
        continue

    for group, names in GROUPS.items():
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for name in names:
            d = h[h["model"] == name].copy()
            if d.empty:
                continue
            d[column] = pd.to_numeric(d[column], errors="coerce")
            d.loc[d[column] <= 0, column] = np.nan
            g = d.groupby("epoch")[column]
            median, q25, q75 = g.median(), g.quantile(.25), g.quantile(.75)
            if median.notna().sum() == 0:
                continue
            line, = ax.plot(median.index, median.values,
                            label=NAMES.get(name, name.replace("mpc_product_k", "Produto k=")))
            ax.fill_between(median.index, q25.values, q75.values,
                            color=line.get_color(), alpha=0.12)
        if ax.lines:
            ax.set_yscale("log")
            ax.set_xlabel("Épocas")
            ax.set_ylabel("Norma L2 (escala log)")
            ax.set_title(f"{title} — {group.upper()}")
            ax.grid(alpha=0.2)
            ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(FIG / f"05_{column}_{group}.png", bbox_inches="tight")
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# Resumo dos gradientes: mediana temporal de cada semente,
# seguida da mediana dessas cinco medianas (sem tratar épocas como sementes independentes).
if {"grad_classical", "grad_aux"}.issubset(h.columns):
    gradient_seed = (
        h.groupby(["model", "seed"], as_index=False)
        .agg(classical_median=("grad_classical", "median"),
             classical_max=("grad_classical", "max"),
             auxiliary_median=("grad_aux", "median"),
             auxiliary_max=("grad_aux", "max"))
    )
    gradient_summary = (
        gradient_seed.groupby("model", as_index=False)
        .agg(n=("seed", "count"),
             classical_median=("classical_median", "median"),
             auxiliary_median=("auxiliary_median", "median"),
             classical_peak_median=("classical_max", "median"),
             auxiliary_peak_median=("auxiliary_max", "median"))
    )
    display(gradient_summary)
    gradient_seed.to_csv(OUT / "gradient_by_seed.csv", index=False)
    gradient_summary.to_csv(OUT / "gradient_summary.csv", index=False)

## 6. MPC com produto de correlatores — influência do peso de Pauli $k$

Aqui $k$ é o **número de operadores diferentes de `I` em cada string de Pauli**, e não o número de correlatores no modelo. Em todos os testes há cinco correlatores e cinco saídas clássicas.

Os conjuntos de operadores também misturam `X`, `Y` e `Z`. Portanto, esta é uma comparação exploratória dos **conjuntos escolhidos**: não atribuímos diferenças exclusivamente ao peso $k$. Se os resultados `mpc_product/k_*/` ainda não existirem, esta seção será pulada automaticamente.

In [ ]:
product = m[m["k"].notna()].copy()
if product.empty:
    print("Ainda não há resultados da MPC com produto. Nada a plotar nesta seção.")
else:
    product["k"] = product["k"].astype(int)
    product_summary = (
        product.groupby("k", as_index=False)
        .agg(n=("seed", "count"), rmse_mean=("rmse", "mean"),
             rmse_median=("rmse", "median"), rmse_q25=("rmse", lambda v: v.quantile(.25)),
             rmse_q75=("rmse", lambda v: v.quantile(.75)),
             l2_median=("relative_l2", "median"))
    )
    display(product_summary)
    product_summary.to_csv(OUT / "pauli_weight_summary.csv", index=False)

    fig, ax = plt.subplots()
    for seed, d in product.groupby("seed"):
        ax.plot(d["k"], d["rmse"], "o-", alpha=0.5, label=f"Seed {seed}")
    ax.plot(product_summary["k"], product_summary["rmse_median"],
            "ko--", linewidth=2.5, label="Mediana")
    ax.set_xticks(range(1, 6))
    ax.set_yscale("log")
    ax.set_xlabel("Peso de Pauli k (não identidades)")
    ax.set_ylabel("RMSE (escala log)")
    ax.set_title("MPC produto — RMSE por peso k")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIG / "06_product_rmse_by_k.png", bbox_inches="tight")
    plt.show()

## 7. Arquivos para montar o relatório

As tabelas `.csv` e as figuras `.png` foram geradas automaticamente em `report/supervised/`. Para preparar o texto final do relatório, sugiro a seguinte ordem:

1. **Método e desenho experimental:** Burgers, arquiteturas, inicialização pareada, dados, sementes, otimizador e scheduler.
2. **Resultados de precisão:** tabela geral e gráficos de RMSE/L2.
3. **Dinâmica de treinamento:** perda por época e limiares de convergência.
4. **Dinâmica dos gradientes:** clássico e auxiliar, com controles congelados bem identificados.
5. **Estudo dos componentes:** comparações pareadas (quântico/clássico, treinável/congelado, potências).
6. **MPC produto:** dependência dos conjuntos de Pauli selecionados pelo peso $k$.
7. **Discussão:** precisão, robustez, otimização e limitações; sem atribuir causalidade quântica apenas pela diferença de erro.

> **Limitação importante:** o histórico contém a perda e os gradientes a cada 100 épocas (além da primeira), mas o RMSE em malha de teste foi salvo somente no final. Não interpretar as curvas de loss como curvas de RMSE de teste.

In [ ]:
print("Tabelas CSV:")
for p in sorted(OUT.glob("*.csv")):
    print(" -", p.relative_to(ROOT))

print(f"\nFiguras PNG: {len(list(FIG.glob('*.png')))}")
print("Pasta:", FIG.relative_to(ROOT))